> 在线讲义：[第 9 章 · 高斯分布：动作头、扩散加噪、卡尔曼滤波都靠它](https://wikieden.github.io/math-and-eval-stats/lessons/09-gaussian.html) · 本书《数学与实验统计》

# 第 9 章 · 高斯分布

讲义：[09-gaussian.md](https://wikieden.github.io/math-and-eval-stats/lessons/09-gaussian.html)。一格一格往下跑；只算讲义里讲过的例子，练习的数自己算。

## 讲义 ↔ 代码

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 | ±kσ | `within_k_sigma` | 0.6827、0.9545、0.9973 |
| 第二步 | ρ、特征分解 | `correlation / ellipse_axes` | 0.849；0.2、2.8 |
| 第三步 | Cholesky、扩散前向 | `sample_via_cholesky / diffusion_forward` | 协方差 ≈ Σ；(1.2, 0.64) |
| 第四步 | 条件分布 | `conditional` | N(1.80, 0.56) |
| 第五步 | log 概率 | `gaussian_log_prob` | 1.384、0.884、−3.116 |

## 定义

In [ ]:
import math
import numpy as np
from scipy.stats import norm
SIGMA = np.array([[2.0, 1.2], [1.2, 1.0]])  # 讲义里的 2×2 协方差（编的）
MU = np.array([0.0, 0.0])

`within_k_sigma`：第一步：一维标准正态落在 ±kσ 内的概率。

In [ ]:
def within_k_sigma(k: float) -> float:
    """第一步：一维标准正态落在 ±kσ 内的概率。"""
    return float(norm.cdf(k) - norm.cdf(-k))

`correlation`：第二步：相关系数 ρ = Σ12 / sqrt(Σ11·Σ22)。

In [ ]:
def correlation(sigma: np.ndarray) -> float:
    """第二步：相关系数 ρ = Σ12 / sqrt(Σ11·Σ22)。"""
    return float(sigma[0, 1] / math.sqrt(sigma[0, 0] * sigma[1, 1]))

`ellipse_axes`：第二步：协方差椭圆 = 特征分解。返回 (半轴长 k·sqrt(λ), 主轴方向)。

In [ ]:
def ellipse_axes(sigma: np.ndarray, k: float = 1.0):
    """第二步：协方差椭圆 = 特征分解。返回 (半轴长 k·sqrt(λ), 主轴方向)。"""
    lam, vec = np.linalg.eigh(sigma)
    vec = vec * np.sign(vec[0])  # 方向向量的正负号是任意的，统一让 x 分量为正
    return k * np.sqrt(lam), vec

`sample_via_cholesky`：第三步：x = μ + L z，L = cholesky(Σ)，z ~ N(0, I)。L Lᵀ = Σ。

In [ ]:
def sample_via_cholesky(mu, sigma, n, seed=0):
    """第三步：x = μ + L z，L = cholesky(Σ)，z ~ N(0, I)。L Lᵀ = Σ。"""
    rng = np.random.default_rng(seed)
    L = np.linalg.cholesky(sigma)
    z = rng.standard_normal((n, len(mu)))
    return mu + z @ L.T

`diffusion_forward`：第三步：扩散前向 x_t = sqrt(ᾱ)·x0 + sqrt(1-ᾱ)·ε 的均值和方差。

In [ ]:
def diffusion_forward(x0: float, alpha_bar: float):
    """第三步：扩散前向 x_t = sqrt(ᾱ)·x0 + sqrt(1-ᾱ)·ε 的均值和方差。"""
    return math.sqrt(alpha_bar) * x0, 1 - alpha_bar

`conditional`：第四步：x1 | x2 的均值 μ1 + Σ12/Σ22·(x2-μ2) 和方差 Σ11 - Σ12²/Σ22。

In [ ]:
def conditional(mu, sigma, x2: float):
    """第四步：x1 | x2 的均值 μ1 + Σ12/Σ22·(x2-μ2) 和方差 Σ11 - Σ12²/Σ22。"""
    m = mu[0] + sigma[0, 1] / sigma[1, 1] * (x2 - mu[1])
    v = sigma[0, 0] - sigma[0, 1] ** 2 / sigma[1, 1]
    return float(m), float(v)

`gaussian_log_prob`：第五步：高斯动作头的 log 概率 = -(a-μ)²/(2σ²) - log σ - ½ log 2π。

In [ ]:
def gaussian_log_prob(a: float, mu: float, sigma: float) -> float:
    """第五步：高斯动作头的 log 概率 = -(a-μ)²/(2σ²) - log σ - ½ log 2π。"""
    return -((a - mu) ** 2) / (2 * sigma**2) - math.log(sigma) - 0.5 * math.log(2 * math.pi)

## 一步一步跑

### 第一步 68-95-99.7

In [ ]:
[round(within_k_sigma(k), 4) for k in (1, 2, 3)]

### 第二步 相关系数与椭圆

In [ ]:
correlation(SIGMA), ellipse_axes(SIGMA)

### 第三步 Cholesky 采样与扩散前向

In [ ]:
np.linalg.cholesky(SIGMA), np.cov(sample_via_cholesky(MU, SIGMA, 200_000).T), diffusion_forward(2, 0.36)

### 第四步 条件分布

In [ ]:
conditional(MU, SIGMA, 1.5)

### 第五步 高斯动作头的 log 概率

In [ ]:
[round(gaussian_log_prob(a, 0.2, 0.1), 3) for a in (0.2, 0.3, 0.5)]

## 代码里才看得到的

1. eigh 的特征值升序，长轴是最后一列；方向正负号任意。
2. 样本按行存，所以写 z @ L.T。
3. np.cov 默认每行一个变量，要传 x.T。

## 自己改一改

- Σ12 改成 −1.2 看方向
- Σ12 改成 1.5 时 cholesky 会报错（不再正定）

In [ ]:
# 在这里改参数试试


## 口诀

> 小误差叠成一口钟，±2σ 九成五；
> 协方差是个椭圆，特征向量定方向；
> 线性变换还是高斯，采样加噪一条线；
> 知道一个缩另一个，单峰平均会撞墙。